In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os
from pathlib import Path

notebook_path = Path().absolute()
sys.path.append(str(notebook_path.parent))

In [3]:
import torch
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from neural_controllers import NeuralController
from utils import newton_dataset
from tqdm import tqdm

from transformers import logging
logging.set_verbosity_error() # Only show errors

torch.manual_seed(0)
torch.cuda.manual_seed(0)
np.random.seed(0)

In [4]:
custom_cache_dir = "/scratch/bbjr/skarmakar/huggingface"

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
model_name='llama_3_8b_it'

language_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto", 
    cache_dir=custom_cache_dir,
)

use_fast_tokenizer = "LlamaForCausalLM" not in language_model.config.architectures
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    use_fast=use_fast_tokenizer, 
    padding_side="left", 
    legacy=False,
)

# tokenizer.pad_token_id = 0 if tokenizer.pad_token_id is None else tokenizer.pad_token_id
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
def find_lingering_forward_hooks(model):
    lingering_hooks = []
    for name, module in model.named_modules():
        if module._forward_hooks:
            hook_info = (name, 'forward')
            lingering_hooks.append(hook_info)
            print(f"Warning: Found {len(module._forward_hooks)} lingering 'forward' hook(s) on module: {name}")

        if module._forward_pre_hooks:
            hook_info = (name, 'forward_pre')
            lingering_hooks.append(hook_info)
            print(f"Warning: Found {len(module._forward_pre_hooks)} lingering 'forward_pre' hook(s) on module: {name}")
            
    if not lingering_hooks:
        print("Success: No lingering forward hooks found in the model.")
        
    return lingering_hooks

In [6]:
rfm_iters = 16
batch_size = 8
n_components = 300
# n_components = 5
# energy = 0.99

In [7]:
path = '../directions/stable/isaac_cam_300'
# path = '../directions/stable/isaac_cam_old'

concept_types = ["Cam", "Isaac"]

controllers = {}

for concept_type in concept_types:
    
    controller = NeuralController(
        language_model,
        tokenizer,
        rfm_iters=rfm_iters,
        batch_size=batch_size,
        control_method='rfm',
        # control_method='logistic',
        n_components=n_components,
        # energy=0.98,
    )
    
    other_type = [k for k in concept_types if k!=concept_type][0]
    
    controller.load(concept=f'{concept_type}', model_name=model_name, path=path)
    
    controllers[concept_type] = controller

n_components: 300
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 16
forward_batch_size   : 8
M_batch_size         : 2048
n_components         : 300

Detector found
n_components: 300
Hidden layers: [-1, -2, -3, -4, -5, -6, -7, -8, -9, -10, -11, -12, -13, -14, -15, -16, -17, -18, -19, -20, -21, -22, -23, -24, -25, -26, -27, -28, -29, -30, -31]

Controller hyperparameters:
control_method       : rfm
rfm_iters            : 16
forward_batch_size   : 8
M_batch_size         : 2048
n_components         : 300

Detector found


/u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/torch/storage.py:414: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(io.BytesIO(b))


In [8]:
# # newton_type = "Isaac"
# newton_type = "Cam"

# raw_inputs = [
#     f"What is Isaac Newton known for?",
#     f"Why is Isaac Newton so famous?",
#     f"What are Isaac Newton's achievements?",
#     # f"Why is Newton the phycisist so famous?",
#     # f"Tell me about newton.",
#     f"Tell me about Isaac Newton.",
#     f"What did Isaac Newton contribute to motion?",
#     # ----------------------------------------
#     f"What is Cam Newton known for?",
#     f"Why is Cam Newton so famous?",
#     f"What are Cam Newton's achievements?",
#     # f"Tell me about the famous sportsman who was the quarterback for the Carolina Panthers of the National Football League (NFL), his surname was Newton.",
#     # f"Why is Newton the sportsman so famous?",
#     f"Tell me about Cam Newton.",
#     f"What did Cam Newton contribute to sports?",
#     # --------------------------------------
#     # f"Tell me about newton.",
#     # f"What is Newton known for?",
#     # f"What is Olivia Newton known for?",
#     # f"What is Huey Newton known for?",
#     "Tell me about ships",
# ]
# inputs = [controller.format_prompt(x) for x in raw_inputs]

# num_new_tokens = 200
# controller = controllers[newton_type]

# # coef=0.4 #llama 
# # coef=0.9
# coef = 1.0
# # coef = 2.0

# # layers = list(range(-5, -31, -1))
# # layers = list(range(-1, -32, -1))
# # layers = list(range(-5, -31, -1))
# # layers = [-3, -4]
# layers = [-6, -7, -8, -9, -10] 

# gens=[]
# print()
# for i in inputs:
#     # print("Prompt:", i)
#     # print("===== No Control =====")
#     # print(controller.generate(
#     #         i, 
#     #         max_new_tokens=num_new_tokens, 
#     #         do_sample=False,
#     #     ).replace(i, "")
#     # )
#     # print()
    
#     # print(f"===== + {newton_type} Control (normal) =====")
#     # gen = controller.generate(
#     #     i, 
#     #     layers_to_control=layers, 
#     #     control_coef=coef, 
#     #     max_new_tokens=num_new_tokens, 
#     #     do_sample=False
#     # ).replace(i, "")
#     # gens.append(gen)

#     # print(f"===== + {newton_type} Control (anti) =====")
#     gen = controller.generate(
#         i, 
#         layers_to_control=layers, 
#         control_coef=coef, 
#         max_new_tokens=num_new_tokens, 
#         do_sample=False,
#         # ----------------------
#         component_idx=200, 
#         anti=True, 
#         last=False,
#         # last=True,
#     # ).replace(i, "")
#     )
#     # gens.append(gen)
#     gens.append("="*50 + "\n" + f"{newton_type} {coef} {layers} "+ "\n" + "="*50 + "\n" + gen)

#     print()
#     print(gens[-1])
#     print()
#     print()

# # for line in gens:
# #     print(f"{line}\n")

/u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/u/skarmakar1/miniconda3/envs/neucon/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(



Cam 1.0 [-6, -7, -8, -9, -10] 
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is Isaac Newton known for?<|eot_id|>
The English mathematician, physicist, and astronomer, who is widely recognized as one of the most influential scientists in history, is known for his work in three main areas: mathematics, physics, and astronomy. He is best known for his laws of motion and his theory of universal gravitation. His most famous work, "Philosophia Naturalis Principia Mathematica" (Mathematical Principles of the Natural World), was published in 1687 and laid the foundation for the scientific revolution of the 17th century. He also made significant contributions to the study of calculus, optics, and the study of the properties of light. His work in mathematics, physics, and astronomy has had a lasting impact on the development of science and technology, and his l

In [9]:
# with open("../outputs/" + "misc" + "/" + str(0) + ".txt", "a") as f:
#     for line in gens:
#         f.write(f"{line}\n")

In [10]:
lh = find_lingering_forward_hooks(language_model)

Success: No lingering forward hooks found in the model.


In [12]:
# newton_type = "Isaac"
newton_type = "Cam"

raw_inputs = [
    f"What is Isaac Newton known for?",
    f"Why is Isaac Newton so famous?",
    f"What are Isaac Newton's achievements?",
    # f"Why is Newton the phycisist so famous?",
    # f"Tell me about newton.",
    f"Tell me about Isaac Newton.",
    f"What did Isaac Newton contribute to motion?",
    # ----------------------------------------
    f"What is Cam Newton known for?",
    f"Why is Cam Newton so famous?",
    f"What are Cam Newton's achievements?",
    # f"Tell me about the famous sportsman who was the quarterback for the Carolina Panthers of the National Football League (NFL), his surname was Newton.",
    # f"Why is Newton the sportsman so famous?",
    f"Tell me about Cam Newton.",
    f"What did Cam Newton contribute to sports?",
    # --------------------------------------
    # f"Tell me about newton.",
    # f"What is Newton known for?",
    # f"What is Olivia Newton known for?",
    # f"What is Huey Newton known for?",
    "Tell me about ships",
    "What is the Central Limit Theorem?",
]
inputs = [controller.format_prompt(x) for x in raw_inputs]

num_new_tokens = 200
controller = controllers[newton_type]

# coef=0.4 #llama 
# coef=0.9


# coef = 1.0
coef = 0.5
# coef = 1.1

top_m = 100

prefill_last_token_only = False


# layers_list = [[-3,-4],]

# layers_list = [list(range(-26, -31, -1)),]

layers_list = [
    list(range(-1, -32, -1)), list(range(-1, -31, -1)), list(range(-1, -6, -1)), list(range(-2, -7, -1)),
    list(range(-6, -11, -1)), list(range(-11, -16, -1)), list(range(-16, -21, -1)), 
    list(range(-21, -26, -1)), list(range(-26, -31, -1)), list(range(-26, -32, -1)), 
]

# layers_list = [
#     list(range(-1, -31, -1)), list(range(-1, -6, -1)), 
#     list(range(-6, -11, -1)), list(range(-11, -16, -1)), list(range(-16, -21, -1)), 
#     list(range(-21, -26, -1)), list(range(-26, -31, -1)),
# ]

# layers_list = [
#     list(range(-1, -11, -1)), list(range(-11, -21, -1)), list(range(-21, -31, -1)), 
# ]

# layers_list = [
#     list(range(-1, -16, -1)), list(range(-16, -32, -1)), 
# ]


# folder_name = "../outputs/" + "5_last"
folder_name = "../outputs/" + f"5_all_m{top_m}_coef{coef}"

os.makedirs(folder_name, exist_ok=True)

ind = 0

for layers in tqdm(layers_list):
    ind += 1
    gens=[]
    for i in inputs:
        # print(f"===== + {newton_type} Control (anti) =====")
        gen = controller.generate(
            i, 
            layers_to_control=layers, 
            control_coef=coef, 
            max_new_tokens=num_new_tokens, 
            do_sample=False,
            # ----------------------
            component_idx=top_m, 
            anti=True, 
            last=prefill_last_token_only,
        )
        # gens.append(gen)
        gens.append("="*50 + "\n" + f"{newton_type}, coef:{coef}, prefill last:{prefill_last_token_only}, {layers} "+ "\n" + "="*50 + "\n" + gen)

    with open(folder_name + "/" + str(ind) + ".txt", "a") as f:
        for line in gens:
            f.write(f"{line}\n")

100%|██████████| 10/10 [13:02<00:00, 78.26s/it]


In [13]:
from scipy.linalg import orth
import numpy as np

In [14]:
c = torch.rand((300, 4096))
h = torch.rand((1, 1, 4096))
hp = torch.rand((1, 38, 4096))

In [15]:
print(c.shape)
print(c[:200].shape)
print(h.shape)
print(hp.shape)

torch.Size([300, 4096])
torch.Size([200, 4096])
torch.Size([1, 1, 4096])
torch.Size([1, 38, 4096])


In [14]:
c_np = c.detach().cpu().numpy() # [300, 4096]
o_np = orth(c_np.T)
o = torch.from_numpy(o_np).to(device=c.device, dtype=c.dtype).contiguous()

print(o.shape)

torch.Size([4096, 300])


In [15]:
print((h @ o).shape)

proj = (h @ o) @ o.T

print(proj.shape)

torch.Size([1, 1, 300])
torch.Size([1, 1, 4096])


In [16]:
h_projected = h - proj

t = h_projected @ o
print(t.shape)
# print(t[0][0])

torch.Size([1, 1, 300])


In [17]:
zeros_tensor = torch.zeros_like(t[0][0])

print(torch.allclose(t[0][0], zeros_tensor, atol=3e-6))

False


In [18]:
l = proj @ o
k = h @ o
# print(l.shape)
# print(l[0][0])
print(torch.allclose(l,k, atol=3e-6))

True


In [19]:
print((hp @ o).shape)

projp = (hp @ o) @ o.T

print(projp.shape)

torch.Size([1, 38, 300])
torch.Size([1, 38, 4096])


In [20]:
h_projectedp = hp - projp

tp = h_projectedp @ o
print(tp.shape)

torch.Size([1, 38, 300])


In [21]:
zeros_tensorp = torch.zeros_like(tp)

print(torch.allclose(tp, zeros_tensorp, atol=8e-6))

True
